# Đọc và làm việc với file Parquet lớn (55.6 triệu dòng)

In [2]:
import os
import time
import pandas as pd
import pyarrow.parquet as pq

# Đường dẫn tới file Parquet đã gộp
PARQUET_PATH = r"D:\Download\E2E\E2E_combined.parquet"

# 1. Xem Metadata của file (Không tải dữ liệu vào RAM, cực nhanh)
parquet_file = pq.ParquetFile(PARQUET_PATH)
print(f"Số dòng trong file Parquet: {parquet_file.metadata.num_rows:,}")
print(f"Số cột trong file Parquet: {parquet_file.metadata.num_columns}")
print("\nDanh sách các cột:")
print(parquet_file.schema.names)
print("\nChi tiết Schema:")
print(parquet_file.schema)

### 2. Đọc thử một vài dòng đầu tiên (Head)
Tải một lượng nhỏ dữ liệu để xem trước nội dung.

In [ ]:
# Đọc 5 dòng đầu tiên
head_table = parquet_file.read_row_group(0).slice(0, 5)
df_head = head_table.to_pandas()
df_head

### 3. Chỉ đọc một số cột nhất định (Khuyên dùng để tiết kiệm RAM)
Vì file rất lớn (55 triệu dòng), nếu bạn chỉ cần phân tích một vài cột, hãy chỉ định tên cột khi đọc để tiết kiệm bộ nhớ.

In [ ]:
# Ví dụ chỉ đọc các cột: date, dc, forecast_qty, Factory_Code
columns_to_read = ['date', 'dc', 'forecast_qty', 'Factory_Code']
df_subset = pd.read_parquet(PARQUET_PATH, columns=columns_to_read)
print(f"Đã tải {len(df_subset):,} dòng với {df_subset.shape[1]} cột.")
df_subset.head()

### 4. Thống kê Số lượng Outlet, Item theo DC và Dòng Tổng cộng (Total)
Chỉ tải các cột cần thiết (`dc`, `outlet_code`, `item_code`) để thống kê:
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng mặt hàng thực tế duy nhất (`So_Item_Unique`)
- Tổng số dòng dữ liệu (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [18]:
start = time.time()

print("Đang đọc dữ liệu các cột 'dc', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...")
df_dc = pd.read_parquet(PARQUET_PATH, columns=['dc', 'outlet_code', 'item_code', 'Case', 'forecast_qty'])

print("Đang tính toán thống kê theo từng DC...")
grouped_dc = df_dc.groupby('dc').agg(
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_Item_Unique=('item_code', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

grouped_dc = grouped_dc.sort_values(by='So_Outlet_Unique', ascending=False).reset_index(drop=True)

print("Đang tính toán dòng tổng cộng (Total) theo DC...")
total_unique_outlets_dc = df_dc['outlet_code'].nunique()
total_unique_items_dc = df_dc['item_code'].nunique()
total_case_dc = df_dc['Case'].sum()
total_forecast_qty_dc = df_dc['forecast_qty'].sum()
total_rows_sum_dc = len(df_dc)

total_row_dc = pd.DataFrame([{
    'dc': 'Total',
    'So_Outlet_Unique': total_unique_outlets_dc,
    'So_Item_Unique': total_unique_items_dc,
    'Tong_Case': total_case_dc,
    'Tong_Forecast_Qty': total_forecast_qty_dc,
    'Tong_So_Dong': total_rows_sum_dc
}])

result_dc_df = pd.concat([grouped_dc, total_row_dc], ignore_index=True)
print(f"Thống kê DC hoàn tất trong {time.time() - start:.1f} giây.")

result_dc_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_Item_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
})

Đang đọc dữ liệu các cột 'dc', 'outlet_code', 'item_code'...
Đang tính toán thống kê theo từng DC...
Đang tính toán dòng tổng cộng (Total) theo DC...
Thống kê DC hoàn tất trong 44.6 giây.


,dc,So_Outlet_Unique,So_Item_Unique,Tong_So_Dong
0,MTA,"82,881",25,"8,729,130"
1,MDQ,"81,181",22,"7,100,580"
2,MTH,"78,079",32,"9,992,940"
3,GMT,"54,317",39,"5,047,200"
4,MTF,"45,507",28,"4,541,250"
5,MTD,"42,300",25,"3,547,290"
6,MTV,"39,261",5,"1,503,090"
7,MTS,"37,282",23,"4,740,120"
8,MDV,"29,548",23,"3,273,990"
9,MDX,"27,127",29,"2,934,870"


### 4.1. Thống kê Số lượng Outlet, Item theo Factory_Code và Dòng Tổng cộng (Total)
Tương tự mục 4 nhưng group by `Factory_Code` thay vì `dc`:
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng mặt hàng thực tế duy nhất (`So_Item_Unique`)
- Tổng Case (`Tong_Case`)
- Tổng Forecast Qty (`Tong_Forecast_Qty`)
- Tổng số dòng dữ liệu (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [ ]:
start = time.time()

print("Dang doc du lieu cac cot 'Factory_Code', 'Factory', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...")
df_factory = pd.read_parquet(PARQUET_PATH, columns=['Factory_Code', 'Factory', 'outlet_code', 'item_code', 'Case', 'forecast_qty'])

print("Dang tinh toan thong ke theo tung Factory_Code...")
grouped_factory = df_factory.groupby('Factory_Code').agg(
    Factory_Name=('Factory', 'first'),
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_Item_Unique=('item_code', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

grouped_factory = grouped_factory.sort_values(by='So_Outlet_Unique', ascending=False).reset_index(drop=True)

print("Dang tinh toan dong tong cong (Total) theo Factory_Code...")
total_row_factory = pd.DataFrame([{
    'Factory_Code': 'Total',
    'Factory_Name': '',
    'So_Outlet_Unique': df_factory['outlet_code'].nunique(),
    'So_Item_Unique': df_factory['item_code'].nunique(),
    'Tong_Case': df_factory['Case'].sum(),
    'Tong_Forecast_Qty': df_factory['forecast_qty'].sum(),
    'Tong_So_Dong': len(df_factory)
}])

result_factory_df = pd.concat([grouped_factory, total_row_factory], ignore_index=True)
print(f"Thong ke Factory_Code hoan tat trong {time.time() - start:.1f} giay.")

result_factory_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_Item_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
})

### 5. Thống kê Số lượng Outlet, Item theo Unit và Dòng Tổng cộng (Total)
Chỉ tải các cột cần thiết (`unit`, `outlet_code`, `item_code`) để thống kê:
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng mặt hàng thực tế duy nhất (`So_Item_Unique`)
- Tổng số dòng dữ liệu (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [ ]:
start = time.time()

print("Đang đọc dữ liệu các cột 'unit', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...")
df_unit = pd.read_parquet(PARQUET_PATH, columns=['unit', 'outlet_code', 'item_code', 'Case', 'forecast_qty'])

print("Đang tính toán thống kê theo từng Unit...")
grouped_unit = df_unit.groupby('unit').agg(
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_Item_Unique=('item_code', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

grouped_unit = grouped_unit.sort_values(by='So_Outlet_Unique', ascending=False).reset_index(drop=True)

print("Đang tính toán dòng tổng cộng (Total) theo Unit...")
total_unique_outlets_unit = df_unit['outlet_code'].nunique()
total_unique_items_unit = df_unit['item_code'].nunique()
total_case_unit = df_unit['Case'].sum()
total_forecast_qty_unit = df_unit['forecast_qty'].sum()
total_rows_sum_unit = len(df_unit)

total_row_unit = pd.DataFrame([{
    'unit': 'Total',
    'So_Outlet_Unique': total_unique_outlets_unit,
    'So_Item_Unique': total_unique_items_unit,
    'Tong_Case': total_case_unit,
    'Tong_Forecast_Qty': total_forecast_qty_unit,
    'Tong_So_Dong': total_rows_sum_unit
}])

result_unit_df = pd.concat([grouped_unit, total_row_unit], ignore_index=True)
print(f"Thống kê Unit hoàn tất trong {time.time() - start:.1f} giây.")

result_unit_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_Item_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
})

### 6. Thống kê Số lượng Outlet, DC theo Item và Dòng Tổng cộng (Total)
Chỉ tải các cột cần thiết (`item_code`, `outlet_code`, `dc`) để thống kê theo từng sản phẩm (`item_code`):
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng DC thực tế phân phối mặt hàng này (`So_DC_Unique`)
- Tổng số dòng giao dịch của mặt hàng (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [17]:
start = time.time()

print("Dang doc du lieu cac cot can thiet cho Item Stats...")
df_item = pd.read_parquet(PARQUET_PATH, columns=[
    'item_code', 'unit', 'outlet_code', 'dc',
    'Case', 'forecast_qty',
    'MD06[UOM1]', 'MD06[UOM2]', 'MD06[UOM Conversion]', 'MD06[Item Name]'
])

print("Dang tinh toan thong ke theo cap (item_code, unit)...")
grouped_item = df_item.groupby(['item_code', 'unit']).agg(
    Item_Name=('MD06[Item Name]', 'first'),
    UOM1=('MD06[UOM1]', 'first'),
    UOM2=('MD06[UOM2]', 'first'),
    UOM_Conversion=('MD06[UOM Conversion]', 'first'),
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_DC_Unique=('dc', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

# Sap xep giam dan theo tong so dong giao dich
grouped_item = grouped_item.sort_values(by=['item_code', 'Tong_So_Dong'], ascending=[True, False]).reset_index(drop=True)

print("Dang tinh toan dong tong cong (Total)...")
total_row_item = pd.DataFrame([{
    'item_code': 'Total',
    'unit': '',
    'Item_Name': '',
    'UOM1': '',
    'UOM2': '',
    'UOM_Conversion': '',
    'So_Outlet_Unique': df_item['outlet_code'].nunique(),
    'So_DC_Unique': df_item['dc'].nunique(),
    'Tong_Case': df_item['Case'].sum(),
    'Tong_Forecast_Qty': df_item['forecast_qty'].sum(),
    'Tong_So_Dong': len(df_item)
}])

result_item_df = pd.concat([grouped_item, total_row_item], ignore_index=True)
print(f"Hoan tat trong {time.time() - start:.1f} giay. So cap (item_code, unit): {len(grouped_item):,}")

# Hien thi bang
display(result_item_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_DC_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
}))

# Xuat ra file Excel
excel_out = r"D:/Download/E2E/Item_Stats_Summary.xlsx"
print(f"Dang xuat ra Excel: {excel_out}")
result_item_df.to_excel(excel_out, index=False, engine='openpyxl')
print(f"Da xuat xong file Excel: {excel_out}")

Đang đọc dữ liệu các cột 'item_code', 'outlet_code', 'dc'...
Đang tính toán thống kê theo từng Item...
Đang tính toán dòng tổng cộng (Total) theo Item...
Thống kê Item hoàn tất trong 28.6 giây.


,item_code,So_Outlet_Unique,So_DC_Unique,Tong_So_Dong
0,08TL00107,"270,538",11,"8,226,060"
1,08TL001,"190,691",16,"6,016,140"
2,02OM00338,"191,261",15,"5,780,100"
3,02OM00770,"169,341",15,"5,123,700"
4,08TL00086,"117,769",15,"3,546,300"
5,02OM00618,"114,767",15,"3,483,750"
6,03NM00789,"106,866",8,"3,217,920"
7,03NM00790,"95,537",5,"2,974,380"
8,03NM00787,"92,154",8,"2,764,620"
9,03NM00788,"91,753",14,"2,756,370"


### 7. Tính số lượng Case (forecast_qty / UOM2/UOM1) và Tạo 2 bảng Pivot (theo Factory_Code và theo item_code)
Logic thực hiện:
- Đọc dữ liệu cột `Item No`, `UOM2/UOM1` từ sheet `MD06` trong file `base line Supra.xlsx` (có cơ chế xử lý chống lỗi khóa file khi file Excel đang mở).
- Đọc các cột `item_code`, `Factory_Code`, `forecast_qty`, `date` từ file Parquet.
- Chuẩn hóa kiểu dữ liệu mã sản phẩm thành chuỗi để thực hiện mapping chính xác.
- Tính toán cột `case` = `forecast_qty` / `UOM2/UOM1`.
- Tạo và hiển thị 2 bảng Pivot riêng biệt:
  1. Bảng 1: Phân tích theo nhà máy (`Factory_Code`)
  2. Bảng 2: Phân tích theo sản phẩm (`item_code`)
- Cả 2 bảng đều hiển thị ngày (`date`) làm cột, có dòng tổng cộng `Total` và định dạng dấu phẩy `,` phân tách hàng ngàn.

In [16]:
start = time.time()

# 1. Đọc sheet MD06 từ file Excel (Bao gồm xử lý chống lỗi file bị khóa)
excel_path = "base line Supra.xlsx"
print(f"Đang đọc sheet 'MD06' từ file '{excel_path}'...")
try:
    MD06 = pd.read_excel(excel_path, sheet_name="MD06")
except PermissionError:
    import shutil
    temp_excel = "temp_base_line_Supra.xlsx"
    print("  -> File đang được mở bởi Excel. Đang tạo bản sao tạm thời để đọc...")
    shutil.copyfile(excel_path, temp_excel)
    MD06 = pd.read_excel(temp_excel, sheet_name="MD06")
    import os
    os.remove(temp_excel)

print(f"Đọc xong MD06. Số dòng: {len(MD06):,}")

# 2. Đọc các cột cần thiết từ file Parquet để tối ưu RAM
print("Đang đọc dữ liệu từ file Parquet (item_code, Factory_Code, forecast_qty, date)...")
columns_to_read = ['item_code', 'Factory_Code', 'forecast_qty', 'date']
E2E_combined = pd.read_parquet(PARQUET_PATH, columns=columns_to_read)
print(f"Đọc xong Parquet. Số dòng: {len(E2E_combined):,}")

# 3. Chuẩn hóa mã sản phẩm thành kiểu chuỗi (String) để tránh lệch kiểu khi mapping
print("Đang chuẩn hóa mã sản phẩm và mapping...")
MD06['Item No'] = MD06['Item No'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
E2E_combined['item_code'] = E2E_combined['item_code'].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)

# Loại bỏ trùng lặp trong MD06 để tránh nhân số dòng khi merge
md06_mapping = MD06[['Item No', 'UOM2/UOM1']].drop_duplicates(subset=['Item No'])

# Merge (Left Join)
merged_df = pd.merge(E2E_combined, md06_mapping, left_on='item_code', right_on='Item No', how='left')

# 4. Tính toán số lượng Case
print("Đang tính số lượng Case...")
merged_df['UOM2/UOM1'] = pd.to_numeric(merged_df['UOM2/UOM1'], errors='coerce')
merged_df['case'] = merged_df['forecast_qty'] / merged_df['UOM2/UOM1']

# 5. Tạo 2 bảng Pivot
# 5.1 Bảng Pivot 1: Theo Factory_Code
print("Đang tính toán bảng Pivot 1 (Factory_Code)...")
pivot_factory = merged_df.pivot_table(
    index='Factory_Code',
    columns='date',
    values='case',
    aggfunc='sum'
)
pivot_factory = pivot_factory.sort_index()
pivot_factory['forecast_qty'] = merged_df.groupby('Factory_Code')['forecast_qty'].sum()
pivot_factory.loc['Total'] = pivot_factory.sum(axis=0)

# 5.2 Bảng Pivot 2: Theo item_code
print("Đang tính toán bảng Pivot 2 (item_code)...")
pivot_item = merged_df.pivot_table(
    index='item_code',
    columns='date',
    values='case',
    aggfunc='sum'
)
pivot_item = pivot_item.sort_index()
pivot_item['forecast_qty'] = merged_df.groupby('item_code')['forecast_qty'].sum()
pivot_item.loc['Total'] = pivot_item.sum(axis=0)

print(f"Hoàn thành tính toán trong {time.time() - start:.1f} giây.")

# 6. Hiển thị 2 bảng Pivot riêng biệt dùng IPython display
from IPython.display import display

print("\n" + "="*40)
print("BẢNG PIVOT 1: TỔNG HỢP CASE THEO FACTORY_CODE")
print("="*40)
display(pivot_factory.style.format('{:,.2f}', na_rep='-'))

print("\n" + "="*40)
print("BẢNG PIVOT 2: TỔNG HỢP CASE THEO ITEM_CODE")
print("="*40)
display(pivot_item.style.format('{:,.2f}', na_rep='-'))

Đang đọc sheet 'MD06' từ file 'base line Supra.xlsx'...
Đọc xong MD06. Số dòng: 22,092
Đang đọc dữ liệu từ file Parquet (item_code, Factory_Code, forecast_qty, date)...
Đọc xong Parquet. Số dòng: 55,622,100
Đang chuẩn hóa mã sản phẩm và mapping...
Đang tính số lượng Case...
Đang tính toán bảng Pivot 1 (Factory_Code)...
Đang tính toán bảng Pivot 2 (item_code)...
Hoàn thành tính toán trong 100.5 giây.

BẢNG PIVOT 1: TỔNG HỢP CASE THEO FACTORY_CODE


date,2026-06-01,2026-06-02,2026-06-03,2026-06-04,2026-06-05,2026-06-06,2026-06-07,2026-06-08,2026-06-09,2026-06-10,2026-06-11,2026-06-12,2026-06-13,2026-06-14,2026-06-15,2026-06-16,2026-06-17,2026-06-18,2026-06-19,2026-06-20,2026-06-21,2026-06-22,2026-06-23,2026-06-24,2026-06-25,2026-06-26,2026-06-27,2026-06-28,2026-06-29,2026-06-30
Factory_Code,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
MBP,"40,035.39","35,231.14","33,629.71","35,231.14","36,832.54","36,832.54","6,405.67","40,035.39","35,231.14","33,629.71","35,231.14","36,832.54","36,832.54","6,405.67","40,035.39","35,231.14","33,629.71","35,231.14","36,832.54","36,832.54","6,405.67","40,035.39","35,231.14","33,629.71","35,231.14","36,832.54","36,832.54","6,405.67","40,035.39","35,231.14"
MDF,0.43,0.38,0.36,0.38,0.40,0.40,0.07,0.43,0.38,0.36,0.38,0.40,0.40,0.07,0.43,0.38,0.36,0.38,0.40,0.40,0.07,0.43,0.38,0.36,0.38,0.40,0.40,0.07,0.43,0.38
MGD,"22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16"
MGP,"96,024.11","84,501.23","80,660.22","84,501.23","88,342.16","88,342.16","15,363.89","96,024.11","84,501.23","80,660.22","84,501.23","88,342.16","88,342.16","15,363.89","96,024.11","84,501.23","80,660.22","84,501.23","88,342.16","88,342.16","15,363.89","96,024.11","84,501.23","80,660.22","84,501.23","88,342.16","88,342.16","15,363.89","96,024.11","84,501.23"
MIP,454.71,400.14,381.95,400.14,418.33,418.33,72.75,454.71,400.14,381.95,400.14,418.33,418.33,72.75,454.71,400.14,381.95,400.14,418.33,418.33,72.75,454.71,400.14,381.95,400.14,418.33,418.33,72.75,454.71,400.14
MPP,0.03,0.02,0.02,0.02,0.03,0.03,0.00,0.03,0.02,0.02,0.02,0.03,0.03,0.00,0.03,0.02,0.02,0.02,0.03,0.03,0.00,0.03,0.02,0.02,0.02,0.03,0.03,0.00,0.03,0.02
PQP,0.17,0.15,0.15,0.15,0.16,0.16,0.03,0.17,0.15,0.15,0.15,0.16,0.16,0.03,0.17,0.15,0.15,0.15,0.16,0.16,0.03,0.17,0.15,0.15,0.15,0.16,0.16,0.03,0.17,0.15
VFF,"22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16","18,546.00","19,429.16","20,312.29","20,312.29","3,532.58","22,078.58","19,429.16"
VFQ,"1,399.06","1,231.17","1,175.21","1,231.17","1,287.13","1,287.13",223.85,"1,399.06","1,231.17","1,175.21","1,231.17","1,287.13","1,287.13",223.85,"1,399.06","1,231.17","1,175.21","1,231.17","1,287.13","1,287.13",223.85,"1,399.06","1,231.17","1,175.21","1,231.17","1,287.13","1,287.13",223.85,"1,399.06","1,231.17"



BẢNG PIVOT 2: TỔNG HỢP CASE THEO ITEM_CODE


date,2026-06-01,2026-06-02,2026-06-03,2026-06-04,2026-06-05,2026-06-06,2026-06-07,2026-06-08,2026-06-09,2026-06-10,2026-06-11,2026-06-12,2026-06-13,2026-06-14,2026-06-15,2026-06-16,2026-06-17,2026-06-18,2026-06-19,2026-06-20,2026-06-21,2026-06-22,2026-06-23,2026-06-24,2026-06-25,2026-06-26,2026-06-27,2026-06-28,2026-06-29,2026-06-30
item_code,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
02OM00338,"17,838.17","16,032.26","15,430.28","16,032.26","16,634.23","16,634.23","5,196.80","17,838.17","16,032.26","15,430.28","16,032.26","16,634.23","16,634.23","5,196.80","17,838.17","16,032.26","15,430.28","16,032.26","16,634.23","16,634.23","5,196.80","17,838.17","16,032.26","15,430.28","16,032.26","16,634.23","16,634.23","5,196.80","17,838.17","16,032.26"
02OM00618,"6,603.50","5,933.84","5,710.62","5,933.84","6,157.06","6,157.06","1,915.87","6,603.50","5,933.84","5,710.62","5,933.84","6,157.06","6,157.06","1,915.87","6,603.50","5,933.84","5,710.62","5,933.84","6,157.06","6,157.06","1,915.87","6,603.50","5,933.84","5,710.62","5,933.84","6,157.06","6,157.06","1,915.87","6,603.50","5,933.84"
02OM00770,"14,090.67","12,673.15","12,200.63","12,673.15","13,145.65","13,145.65","4,167.98","14,090.67","12,673.15","12,200.63","12,673.15","13,145.65","13,145.65","4,167.98","14,090.67","12,673.15","12,200.63","12,673.15","13,145.65","13,145.65","4,167.98","14,090.67","12,673.15","12,200.63","12,673.15","13,145.65","13,145.65","4,167.98","14,090.67","12,673.15"
03HH00033,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
03NM00523,"3,861.83","3,465.80","3,333.79","3,465.80","3,597.81","3,597.81","1,089.62","3,861.83","3,465.80","3,333.79","3,465.80","3,597.81","3,597.81","1,089.62","3,861.83","3,465.80","3,333.79","3,465.80","3,597.81","3,597.81","1,089.62","3,861.83","3,465.80","3,333.79","3,465.80","3,597.81","3,597.81","1,089.62","3,861.83","3,465.80"
03NM00532,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01
03NM00535,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.01,0.01,0.01,0.01,0.00,0.01,0.01
03NM00597,0.35,0.31,0.29,0.31,0.32,0.32,0.06,0.35,0.31,0.29,0.31,0.32,0.32,0.06,0.35,0.31,0.29,0.31,0.32,0.32,0.06,0.35,0.31,0.29,0.31,0.32,0.32,0.06,0.35,0.31
03NM00601,"5,827.05","5,214.90","5,010.85","5,214.90","5,418.95","5,418.95","1,542.02","5,827.05","5,214.90","5,010.85","5,214.90","5,418.95","5,418.95","1,542.02","5,827.05","5,214.90","5,010.85","5,214.90","5,418.95","5,418.95","1,542.02","5,827.05","5,214.90","5,010.85","5,214.90","5,418.95","5,418.95","1,542.02","5,827.05","5,214.90"


### 7.1. Kiểm tra các mã sản phẩm (item_code) trong Parquet bị thiếu trong Master Data MD06
Tìm các mã sản phẩm xuất hiện trong dữ liệu giao dịch (`E2E_combined`) nhưng không có thông tin quy đổi ở Master Data (`MD06`). Việc thiếu mã này sẽ dẫn đến lượng `case` bị tính là `NaN` hoặc lỗi.

In [ ]:
# Lấy tập hợp mã unique của cả 2 bảng (sử dụng dữ liệu đã được chuẩn hóa chuỗi ở cell trên)
unique_parquet_items = set(E2E_combined['item_code'].unique())
unique_md06_items = set(MD06['Item No'].unique())

# Tìm phần giao lệch (những mã trong Parquet mà không có trong MD06)
missing_in_md06 = unique_parquet_items - unique_md06_items

print(f"Tổng số mã sản phẩm duy nhất trong Parquet (E2E_combined): {len(unique_parquet_items):,}")
print(f"Tổng số mã sản phẩm duy nhất trong Master Data (MD06): {len(unique_md06_items):,}")
print(f"Số lượng mã bị thiếu trong MD06: {len(missing_in_md06):,}")

if len(missing_in_md06) > 0:
    print("\nDanh sách các mã sản phẩm bị thiếu trong MD06:")
    # Hiển thị sắp xếp theo thứ tự chữ cái
    for code in sorted(list(missing_in_md06)):
        print(f"- {code}")
else:
    print("\nChúc mừng! Tất cả mã sản phẩm trong file Parquet đều có đầy đủ thông tin trong MD06.")

### 8. Đọc toàn bộ file vào Pandas DataFrame (Cần RAM lớn)
Hãy chắc chắn máy tính của bạn còn đủ RAM trống (khuyên dùng máy >= 16GB RAM).

In [ ]:
import time
start = time.time()
print("Đang tải toàn bộ 55.6 triệu dòng vào RAM...")
df = pd.read_parquet(PARQUET_PATH)
print(f"Đã tải xong trong {time.time() - start:.1f} giây. Bộ nhớ DataFrame chiếm dụng: {df.memory_usage().sum() / 1024**2:.1f} MB")
df.head()

### 9. Đọc bằng Polars (Nếu đã cài đặt, tối ưu hiệu năng và hỗ trợ Lazy Evaluation)
Polars xử lý dữ liệu lớn cực kỳ nhanh và có chế độ `scan_parquet` (Lazy loading) giúp bạn truy vấn dữ liệu lớn mà không tốn RAM.

In [ ]:
try:
    import polars as pl
    # Lazy loading
    lf = pl.scan_parquet(PARQUET_PATH)
    
    print("--- THỐNG KÊ THEO DC BẰNG POLARS ---")
    summary_dc = lf.group_by("dc").agg([
        pl.col("outlet_code").n_unique().alias("So_Outlet_Unique"),
        pl.col("item_code").n_unique().alias("So_Item_Unique"),
        pl.col("Case").sum().alias("Tong_Case"),
        pl.col("forecast_qty").sum().alias("Tong_Forecast_Qty"),
        pl.col("outlet_code").count().alias("Tong_So_Dong")
    ]).sort("So_Outlet_Unique", descending=True).collect()
    print(summary_dc)
    
    print("\n--- THỐNG KÊ THEO UNIT BẰNG POLARS ---")
    summary_unit = lf.group_by("unit").agg([
        pl.col("outlet_code").n_unique().alias("So_Outlet_Unique"),
        pl.col("item_code").n_unique().alias("So_Item_Unique"),
        pl.col("Case").sum().alias("Tong_Case"),
        pl.col("forecast_qty").sum().alias("Tong_Forecast_Qty"),
        pl.col("outlet_code").count().alias("Tong_So_Dong")
    ]).sort("So_Outlet_Unique", descending=True).collect()
    print(summary_unit)

    print("\n--- THỐNG KÊ THEO ITEM BẰNG POLARS ---")
    summary_item = lf.group_by("item_code").agg([
        pl.col("outlet_code").n_unique().alias("So_Outlet_Unique"),
        pl.col("dc").n_unique().alias("So_DC_Unique"),
        pl.col("Case").sum().alias("Tong_Case"),
        pl.col("forecast_qty").sum().alias("Tong_Forecast_Qty"),
        pl.col("outlet_code").count().alias("Tong_So_Dong")
    ]).sort("Tong_So_Dong", descending=True).collect()
    print(summary_item)
except ImportError:
    print("Chưa cài đặt thư viện 'polars'. Bạn có thể cài đặt bằng lệnh: pip install polars")